# 03b · RoBERTa penultimate embeddings — a richer sentiment representation

**Why both the score vector and the embedding come from the same place.** The 3-D `[pos,neg,neu]` score
is the model's *final sentiment decision*; the 768-d embedding is the encoder representation the model
*built in order to make that decision*. Because Twitter-RoBERTa is **fine-tuned for sentiment**, that
representation is organized around sentiment — it is not a general semantic embedding, it is a
sentiment-shaped one. So it is a legitimate sentiment representation, just richer than the score.

**When the score vector makes more sense:** when the claim is specifically about *valence/sentiment* —
it is the purest, most interpretable object, and it is exactly what Step 1 validated against behavior
(cv-R² 0.34). Use it as the headline.

**When the embedding makes more sense:** when you need a richer, higher-**reliability** representation
comparable to Jin's 512-d USE vectors — the 3-D score's run-split reliability was only ~0.03, which is
what capped the 04b IS-RSA. The embedding may lift you off that noise floor. *Caveat:* it still encodes
the content that predicts sentiment, so a brain effect reads as 'tracks the sentiment representation,'
slightly broader than 'tracks valence.' Report it as the reliability-motivated comparison, not a replacement.

This notebook (1) extracts the embeddings, (2) builds the same-format similarity matrices, (3) checks
whether reliability actually improves over the 3-D score.

In [1]:
import pandas as pd, numpy as np, torch, pickle, os
from transformers import AutoTokenizer, AutoModel
MODEL_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"   # SAME model as Step 0 winner
tok = AutoTokenizer.from_pretrained(MODEL_ID)
enc = AutoModel.from_pretrained(MODEL_ID)   # base encoder (fine-tuned weights), no classification head
enc.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"; enc.to(device)

sc = pd.read_csv("results/scored/00__reflection_sentiment.csv")
sc["Character"] = sc["Character"].str.lower().str.strip()
meta = sc.dropna(subset=["Raw_Text"]).drop_duplicates(["Participant","Run","Character"])
print(len(meta), "reflections to embed")

@torch.no_grad()
def embed(text):
    e = tok(str(text), return_tensors="pt", truncation=True, max_length=512).to(device)
    h = enc(**e).last_hidden_state[0]                 # (tokens, 768) penultimate activations
    m = e["attention_mask"][0].unsqueeze(-1)
    return ((h*m).sum(0) / m.sum().clamp(min=1)).cpu().numpy()   # attention-masked mean pool

embs = {}
for k,(_,r) in enumerate(meta.iterrows()):
    embs[(r.Participant, int(r.Run), r.Character)] = embed(r.Raw_Text)
    if k % 200 == 0: print(f"  {k}/{len(meta)}")
os.makedirs("results/embeddings", exist_ok=True)
pickle.dump(embs, open("results/embeddings/03b__roberta_embeddings.pkl","wb"))
print("saved", len(embs), "embeddings (768-d)")

/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1272 reflections to embed
  0/1272
  200/1272
  400/1272
  600/1272
  800/1272
  1000/1272
  1200/1272
saved 1272 embeddings (768-d)


## Build between-subject similarity (same format as `03__sentiment_sim_byrun_bychar.npy`) + reliability check

In [2]:
import numpy as np, json as _json, pickle
from scipy.stats import spearmanr
emb = pickle.load(open("results/embeddings/03b__roberta_embeddings.pkl","rb"))
overlap = _json.load(open("results/jinrep/03__isrsa_subject_order.json"))   # 29 overlap, Jin order
CHAR = ["jack","kate","randall","kevin"]
def cos(a,b):
    if a is None or b is None: return np.nan
    d = np.linalg.norm(a)*np.linalg.norm(b); return float(np.dot(a,b)/d) if d else np.nan
sim = {}
for g in [1,2,3]:
    subs = overlap[str(g)]
    for run in range(1,11):
        chs = []
        for ch in CHAR:
            v = [emb.get((s,run,ch)) for s in subs]
            chs.append([cos(v[i],v[j]) for i in range(len(v)) for j in range(i+1,len(v))])
        sim[g,run] = np.array(chs)                      # (4 char, n_pairs) — 29-overlap, Jin order
np.save("results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy", sim, allow_pickle=True)
print("saved results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy (same format as the 3-D sentiment file)")

# reliability: odd vs even runs split-half of the RDM (same metric that gave the 3-D score ~0.03)
odd  = np.concatenate([sim[g,r].ravel() for g in [1,2,3] for r in [1,3,5,7,9]])
even = np.concatenate([sim[g,r].ravel() for g in [1,2,3] for r in [2,4,6,8,10]])
m = ~(np.isnan(odd)|np.isnan(even))
print(f"embedding RDM split-half (odd vs even runs): {spearmanr(odd[m],even[m])[0]:.3f}   (3-D score was ~0.03)")

saved results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy (same format as the 3-D sentiment file)
embedding RDM split-half (odd vs even runs): 0.005   (3-D score was ~0.03)


## Run the IS-RSA on the embedding

In `04b`, point the sentiment-similarity loader at this file instead of the 3-D one:

```python
SENT = np.load("results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy", allow_pickle=True).item()
```

then re-run `isrsa()` and the dual-readout cell. Compare its regions (and the reliability above) to the
3-D score run — if the embedding clears the reliability floor and finds regions the 3-D score didn't,
that's the reliability story; if it just recapitulates Jin's USE result, the Step-3 RSA (USE vs RoBERTa)
will have already told you they share geometry.